# U-Net aplicada ao uso e cobertura da terra
## Bacia Pirajibu-Mirim — Sentinel-2 (2025)

**Objetivo:** classificar o uso e a cobertura da terra da bacia com uma U-Net, usando a mesma imagem Sentinel-2 da aula de Random Forest e, como rótulo de treino, o MapBiomas estável 2021–2025.

**Arquivo necessário (pasta `GEE_Pirajibu_Mirim` no Google Drive):** `UNet_pilha_S2_MapBiomas_2025.tif`, gerado pelo script `GEE_UNet_preparacao_Pirajibu.js`.

**Antes de começar:** `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)`.

**Etapas:**
1. Instalação e configuração
2. Leitura e visualização dos dados
3. Divisão espacial em blocos (treino × validação)
4. Recortes (tiles) e aumento de dados
5. Construção da U-Net
6. Treinamento
7. Mapa da bacia (inferência por janela deslizante)
8. Avaliação do modelo e área por classe
9. Fechamento: qual método é melhor?

## 1. Instalação e configuração

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle

import rasterio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

from sklearn.metrics import confusion_matrix, cohen_kappa_score

In [ ]:
# ---------------- CAMINHOS ----------------
PASTA      = '/content/drive/MyDrive/GEE_Pirajibu_Mirim'
ARQ_PILHA  = f'{PASTA}/UNet_pilha_S2_MapBiomas_2025.tif'
ARQ_PESOS  = f'{PASTA}/unet_pirajibu_melhor.pth'          # pesos da U-Net treinada
ARQ_MAPA   = f'{PASTA}/UNet_LULC_2025_Pirajibu.tif'       # mapa final
ARQ_CONF   = f'{PASTA}/UNet_confianca_2025_Pirajibu.tif'  # probabilidade da classe escolhida

# ---------------- CLASSES ----------------
NOMES = ['Floresta', 'Vegetação herbácea', 'Silvicultura',
         'Agropecuária', 'Área não vegetada', 'Água']
CORES = ['#1f8d49', '#d6bc74', '#7a5900', '#ffefc3', '#d4271e', '#2532e4']
BANDAS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI']
N_BANDAS, N_CLASSES = len(BANDAS), len(NOMES)
IGNORAR = 255                 # valor dos pixels sem rótulo

# ---------------- PARÂMETROS ----------------
TREINAR      = True    # False = só carrega os pesos já treinados (ARQ_PESOS)
TAM          = 128     # tamanho do recorte (pixels) -> 128 x 10 m = 1,28 km
PASSO        = 64      # deslocamento entre recortes (sobreposição de 50%)
BLOCO        = 384     # tamanho dos blocos espaciais da divisão treino/validação
FRAC_VAL     = 0.2     # fração dos blocos usada na validação
MIN_ROTULADO = 0.3     # recorte só entra se tiver >= 30% de pixels rotulados
EPOCAS       = 40
LOTE         = 16
TAXA_APREND  = 5e-4
PESOS_IMAGENET = 'imagenet'   # codificador pré-treinado (transfer learning)
SEMENTE      = 42

np.random.seed(SEMENTE); torch.manual_seed(SEMENTE)
DISP = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Processando em:', DISP)

cmap_classes = ListedColormap(CORES)
legenda_classes = [Patch(color=c, label=f'{i+1} – {n}') for i, (c, n) in enumerate(zip(CORES, NOMES))]

## 2. Leitura e visualização dos dados

A pilha exportada do GEE tem 11 bandas:
- **1–9:** atributos (B2, B3, B4, B8, B11, B12, NDVI, NDWI, NDBI);
- **10:** rótulo do MapBiomas estável (0 = sem rótulo, 1–6 = classes);
- **11:** máscara da bacia (1 = dentro).

> Se o arquivo exportado pelo GEE vier com um sufixo no nome (ex.: `...-0000000000-0000000000.tif`), ajuste `ARQ_PILHA`.

In [ ]:
with rasterio.open(ARQ_PILHA) as src:
    pilha = np.nan_to_num(src.read().astype('float32'))
    PERFIL, TRANSF, CRS = src.profile, src.transform, src.crs

X       = pilha[:N_BANDAS]                 # atributos
rotulo  = pilha[N_BANDAS].astype(np.int64) # 0..6
bacia   = pilha[N_BANDAS + 1] > 0
valido  = np.any(X != 0, axis=0)           # pixel com dado de imagem
H, W = rotulo.shape
print(f'Tamanho: {H} x {W} pixels ({H*10/1000:.1f} x {W*10/1000:.1f} km) | SRC: {CRS}')

# Rótulos 1..6 -> 0..5 (formato do PyTorch); sem rótulo -> IGNORAR
y = np.full((H, W), IGNORAR, dtype=np.int64)
m = (rotulo >= 1) & (rotulo <= N_CLASSES) & valido
y[m] = rotulo[m] - 1
print(f'Pixels rotulados: {m.mean()*100:.1f}% da área exportada')

# Normalização por banda (média 0, desvio 1) calculada nos pixels válidos
MEDIA  = X[:, valido].mean(axis=1)
DESVIO = X[:, valido].std(axis=1) + 1e-6
Xn = ((X - MEDIA[:, None, None]) / DESVIO[:, None, None]).astype('float32')
Xn[:, ~valido] = 0

pd.DataFrame({'banda': BANDAS, 'média': MEDIA.round(4), 'desvio': DESVIO.round(4)})

In [ ]:
def rgb(X, r=2, g=1, b=0, maximo=0.15):
    return np.clip(np.dstack([X[r], X[g], X[b]]) / maximo, 0, 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(rgb(X)); ax[0].contour(bacia, levels=[0.5], colors='red', linewidths=1)
ax[0].set_title('Sentinel-2 2025 (cor natural) + limite da bacia')
ax[1].imshow(np.ma.masked_where(rotulo == 0, rotulo), cmap=cmap_classes, vmin=1, vmax=N_CLASSES, interpolation='nearest')
ax[1].contour(bacia, levels=[0.5], colors='black', linewidths=1)
ax[1].set_title('Rótulo de treino: MapBiomas estável 2021–2025\n(branco = sem rótulo, ignorado no treino)')
ax[1].legend(handles=legenda_classes, loc='lower left', fontsize=8)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

cont = np.bincount(y[y != IGNORAR], minlength=N_CLASSES)
pd.DataFrame({'classe': NOMES, 'pixels rotulados': cont, '%': (cont / cont.sum() * 100).round(1)})

## 3. Divisão espacial em blocos

Recortes vizinhos são muito parecidos (**autocorrelação espacial**). Se sortearmos recortes ao acaso, recortes quase iguais caem no treino e na validação, e o desempenho parece melhor do que é.

Por isso dividimos a área em **blocos**, e cada bloco inteiro vai para o treino **ou** para a validação. Os blocos de validação servem para acompanhar o treinamento e, no fim, para avaliar o mapa em áreas que a rede **nunca viu**.

In [ ]:
rng = np.random.default_rng(SEMENTE)
nby, nbx = int(np.ceil(H / BLOCO)), int(np.ceil(W / BLOCO))
blocos = [(by, bx) for by in range(nby) for bx in range(nbx)]
n_val = max(1, int(round(FRAC_VAL * len(blocos))))
blocos_val = set(blocos[i] for i in rng.permutation(len(blocos))[:n_val])

def recortes_do_bloco(by, bx):
    y0, x0 = by * BLOCO, bx * BLOCO
    y1, x1 = min(y0 + BLOCO, H), min(x0 + BLOCO, W)
    pos = []
    for r in range(y0, y1 - TAM + 1, PASSO):
        for c in range(x0, x1 - TAM + 1, PASSO):
            if (y[r:r+TAM, c:c+TAM] != IGNORAR).mean() >= MIN_ROTULADO and valido[r:r+TAM, c:c+TAM].mean() >= 0.9:
                pos.append((r, c))
    return pos

pos_treino, pos_val = [], []
for b in blocos:
    (pos_val if b in blocos_val else pos_treino).extend(recortes_do_bloco(*b))

print(f'Blocos: {len(blocos)} ({len(blocos) - n_val} treino, {n_val} validação)')
print(f'Recortes: {len(pos_treino)} treino | {len(pos_val)} validação')
if len(pos_val) == 0:
    print('ATENÇÃO: nenhum recorte de validação. Mude SEMENTE ou diminua BLOCO.')

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb(X))
for (by, bx) in blocos:
    cor = 'orange' if (by, bx) in blocos_val else 'cyan'
    ax.add_patch(Rectangle((bx*BLOCO, by*BLOCO), BLOCO, BLOCO, fill=False, edgecolor=cor, lw=2))
ax.contour(bacia, levels=[0.5], colors='red', linewidths=1)
ax.legend(handles=[Patch(edgecolor='cyan', fill=False, label='treino'),
                   Patch(edgecolor='orange', fill=False, label='validação')], loc='lower left')
ax.set_title('Divisão espacial em blocos'); ax.axis('off'); plt.show()

## 4. Recortes (tiles) e aumento de dados

Cada exemplo de treino é um par **recorte da imagem (9 × 128 × 128)** + **máscara (128 × 128)**.

**Aumento de dados:** rotações de 90° e espelhamentos. São válidos porque a imagem é vista de cima: não existe "lado certo". Não alteramos cores, para não distorcer o significado espectral.

In [ ]:
class Recortes(Dataset):
    def __init__(self, posicoes, aumentar=False):
        self.pos, self.aumentar = posicoes, aumentar
    def __len__(self):
        return len(self.pos)
    def __getitem__(self, i):
        r, c = self.pos[i]
        x = Xn[:, r:r+TAM, c:c+TAM]
        yy = y[r:r+TAM, c:c+TAM]
        if self.aumentar:
            k = np.random.randint(4)
            x, yy = np.rot90(x, k, axes=(1, 2)), np.rot90(yy, k)
            if np.random.rand() < 0.5:
                x, yy = x[:, :, ::-1], yy[:, ::-1]
        return torch.from_numpy(x.copy()), torch.from_numpy(yy.copy())

dl_treino = DataLoader(Recortes(pos_treino, aumentar=True), batch_size=LOTE, shuffle=True, drop_last=len(pos_treino) > LOTE)
dl_val    = DataLoader(Recortes(pos_val), batch_size=LOTE)

# Exemplos de pares recorte + máscara
n_ex = min(4, len(pos_treino))
fig, ax = plt.subplots(2, n_ex, figsize=(3.5 * n_ex, 7), squeeze=False)
for j, i in enumerate(np.random.choice(len(pos_treino), n_ex, replace=False)):
    r, c = pos_treino[i]
    ax[0, j].imshow(rgb(X[:, r:r+TAM, c:c+TAM]))
    ax[1, j].imshow(np.ma.masked_equal(y[r:r+TAM, c:c+TAM], IGNORAR), cmap=cmap_classes, vmin=0, vmax=N_CLASSES-1, interpolation='nearest')
    ax[0, j].set_title(f'Recorte {i}'); ax[0, j].axis('off'); ax[1, j].axis('off')
ax[1, 0].set_ylabel('máscara')
plt.suptitle('O que a U-Net recebe: recorte (em cima) e máscara (embaixo)'); plt.tight_layout(); plt.show()

## 5. Construção da U-Net

- **Codificador:** ResNet-34 pré-treinada (ImageNet). Como a entrada tem 9 bandas e não 3, a biblioteca adapta automaticamente a primeira camada.
- **Decodificador:** caminho de expansão com *skip connections*.
- **Saída:** 6 mapas (um por classe) → softmax por pixel.

**Função de perda:** entropia cruzada **ponderada** (classes raras, como água, pesam mais) e **ignorando** os pixels sem rótulo.

In [ ]:
modelo = smp.Unet(encoder_name='resnet34', encoder_weights=PESOS_IMAGENET,
                  in_channels=N_BANDAS, classes=N_CLASSES).to(DISP)
print(f'Parâmetros treináveis: {sum(p.numel() for p in modelo.parameters() if p.requires_grad)/1e6:.1f} milhões')

# pesos das classes: inverso da raiz da frequência nos recortes de treino
freq = np.zeros(N_CLASSES)
for r, c in pos_treino:
    yy = y[r:r+TAM, c:c+TAM]
    freq += np.bincount(yy[yy != IGNORAR], minlength=N_CLASSES)
peso_cls = 1 / np.sqrt(np.maximum(freq, 1))
peso_cls = peso_cls / peso_cls.mean()
print(pd.DataFrame({'classe': NOMES, 'pixels no treino': freq.astype(int), 'peso na perda': peso_cls.round(2)}))

perda = nn.CrossEntropyLoss(weight=torch.tensor(peso_cls, dtype=torch.float32, device=DISP), ignore_index=IGNORAR)

## 6. Treinamento

A cada época a rede vê todos os recortes de treino e é avaliada nos recortes de validação. Guardamos os pesos da **melhor época** (menor perda na validação).

- Perda de treino caindo e de validação **subindo** = **sobreajuste**.
- Em sala, use `TREINAR = False` para carregar os pesos já treinados, ou `EPOCAS = 3` para ver o treino acontecendo.

In [ ]:
def rodar_epoca(dl, treinar):
    modelo.train(treinar)
    soma, n, acertos, total = 0.0, 0, 0, 0
    with torch.set_grad_enabled(treinar):
        for xb, yb in dl:
            xb, yb = xb.to(DISP), yb.to(DISP)
            saida = modelo(xb)
            loss = perda(saida, yb)
            if treinar:
                otim.zero_grad(); loss.backward(); otim.step()
            soma += loss.item() * len(xb); n += len(xb)
            ok = yb != IGNORAR
            acertos += (saida.argmax(1)[ok] == yb[ok]).sum().item(); total += ok.sum().item()
    return soma / max(n, 1), acertos / max(total, 1)

historico = []
if TREINAR:
    otim = torch.optim.Adam(modelo.parameters(), lr=TAXA_APREND)
    melhor = np.inf
    for ep in range(1, EPOCAS + 1):
        t0 = time.time()
        pt, at = rodar_epoca(dl_treino, True)
        pv, av = rodar_epoca(dl_val, False) if len(pos_val) else (np.nan, np.nan)
        historico.append((ep, pt, pv, at, av))
        marca = ''
        if len(pos_val) == 0 or pv < melhor:
            melhor = pv if len(pos_val) else melhor
            torch.save({'pesos': modelo.state_dict(), 'media': MEDIA, 'desvio': DESVIO,
                        'bandas': BANDAS, 'nomes': NOMES, 'epoca': ep}, ARQ_PESOS)
            marca = '  <- melhor (salvo)'
        print(f'Época {ep:3d} | perda treino {pt:.3f} val {pv:.3f} | acerto treino {at:.3f} val {av:.3f} | {time.time()-t0:.0f}s{marca}')

ckpt = torch.load(ARQ_PESOS, map_location=DISP, weights_only=False)
modelo.load_state_dict(ckpt['pesos'])
print(f"Pesos carregados (época {ckpt['epoca']}).")

if historico:
    h = pd.DataFrame(historico, columns=['época', 'perda_treino', 'perda_val', 'acerto_treino', 'acerto_val'])
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    h.plot(x='época', y=['perda_treino', 'perda_val'], ax=ax[0], marker='o', ms=3); ax[0].set_title('Perda (loss)')
    h.plot(x='época', y=['acerto_treino', 'acerto_val'], ax=ax[1], marker='o', ms=3); ax[1].set_title('Acerto por pixel (concordância com o rótulo)')
    plt.tight_layout(); plt.show()

## 7. Mapa da bacia (inferência por janela deslizante)

A área é maior que o recorte, então o modelo percorre a imagem em **janelas sobrepostas**. Nas sobreposições, as probabilidades são combinadas com peso maior no centro da janela, onde a rede tem mais contexto. Isso evita as **linhas de emenda**.

Geramos dois produtos:
- **mapa de classes** (classe de maior probabilidade);
- **mapa de confiança** (probabilidade da classe escolhida): valores baixos indicam onde o modelo está em dúvida.

In [ ]:
def prever(Xn, tam=256, passo=128):
    C, Hh, Ww = Xn.shape
    ph, pw = max(0, tam - Hh), max(0, tam - Ww)
    Xp = np.pad(Xn, ((0, 0), (0, ph), (0, pw)), mode='reflect') if (ph or pw) else Xn
    _, Hp, Wp = Xp.shape
    def posicoes(n):
        p = list(range(0, n - tam + 1, passo))
        if p[-1] + tam < n: p.append(n - tam)
        return p
    janela = np.outer(np.hanning(tam), np.hanning(tam)) + 1e-3
    prob = np.zeros((N_CLASSES, Hp, Wp), 'float32'); peso = np.zeros((Hp, Wp), 'float32')
    modelo.eval()
    with torch.no_grad():
        for r in posicoes(Hp):
            for c in posicoes(Wp):
                xb = torch.from_numpy(Xp[None, :, r:r+tam, c:c+tam].copy()).to(DISP)
                p = torch.softmax(modelo(xb), 1)[0].cpu().numpy()
                prob[:, r:r+tam, c:c+tam] += p * janela
                peso[r:r+tam, c:c+tam] += janela
    prob = prob / peso
    return prob[:, :Hh, :Ww]

prob = prever(Xn)
mapa_unet = (prob.argmax(0) + 1).astype('uint8')
confianca = prob.max(0)
dentro = bacia & valido
mapa_unet[~dentro] = 0
confianca[~dentro] = -1

perfil = PERFIL.copy(); perfil.update(count=1, dtype='uint8', nodata=0, compress='lzw')
with rasterio.open(ARQ_MAPA, 'w', **perfil) as dst:
    dst.write(mapa_unet, 1)
perfil.update(dtype='float32', nodata=-1)
with rasterio.open(ARQ_CONF, 'w', **perfil) as dst:
    dst.write(confianca.astype('float32'), 1)
print('Salvos:', ARQ_MAPA, '|', ARQ_CONF)

In [ ]:
def recorte_bacia(a):
    lin, col = np.where(bacia)
    return a[..., lin.min():lin.max()+1, col.min():col.max()+1]

fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].imshow(recorte_bacia(np.moveaxis(rgb(X), -1, 0)).transpose(1, 2, 0))
ax[0].set_title('Sentinel-2 2025')
ax[1].imshow(recorte_bacia(np.ma.masked_equal(np.where(bacia, rotulo, 0), 0)), cmap=cmap_classes, vmin=1, vmax=N_CLASSES, interpolation='nearest')
ax[1].set_title('MapBiomas estável (rótulo de treino)')
ax[2].imshow(recorte_bacia(np.ma.masked_equal(mapa_unet, 0)), cmap=cmap_classes, vmin=1, vmax=N_CLASSES, interpolation='nearest')
ax[2].set_title('U-Net 2025')
ax[2].legend(handles=legenda_classes, loc='lower left', fontsize=8)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

fig, a = plt.subplots(figsize=(7, 7))
im = a.imshow(np.ma.masked_less(recorte_bacia(confianca), 0), cmap='magma', vmin=0.3, vmax=1)
plt.colorbar(im, ax=a, shrink=0.7, label='probabilidade da classe escolhida')
a.set_title('U-Net – confiança (escuro = dúvida)'); a.axis('off'); plt.show()

## 8. Avaliação do modelo e área por classe

Avaliamos o mapa nos **blocos de validação**, áreas que a rede não usou para aprender, comparando com o MapBiomas estável. Usamos a mesma lógica da aula de validação: matriz de confusão, acurácia global, Kappa e acurácias do produtor e do usuário.

> **Atenção:** a referência aqui é o próprio MapBiomas, a mesma fonte do rótulo de treino. Por isso o resultado mede **concordância com o MapBiomas**, não acurácia em relação à realidade. Para medir acurácia de verdade, seria preciso uma amostra independente (fotointerpretação ou campo).

In [ ]:
mascara_val = np.zeros((H, W), bool)
for (by, bx) in blocos_val:
    mascara_val[by*BLOCO:(by+1)*BLOCO, bx*BLOCO:(bx+1)*BLOCO] = True

pred = prob.argmax(0) + 1
aval = mascara_val & valido & (rotulo > 0)
ref, pr = rotulo[aval], pred[aval]
rotulos = list(range(1, N_CLASSES + 1))

cm = confusion_matrix(ref, pr, labels=rotulos)
ag = np.trace(cm) / cm.sum()
kappa = cohen_kappa_score(ref, pr, labels=rotulos)
with np.errstate(divide='ignore', invalid='ignore'):
    prod, usu = np.diag(cm) / cm.sum(1), np.diag(cm) / cm.sum(0)

print(f'Pixels avaliados (blocos de validação): {aval.sum():,}')
print(f'Acurácia global: {ag:.3f} | Kappa: {kappa:.3f}')
display(pd.DataFrame({'Acurácia do produtor': prod, 'Acurácia do usuário': usu,
                      'Pixels de referência': cm.sum(1)}, index=NOMES).round(3))

fig, a = plt.subplots(figsize=(7, 6))
a.imshow(cm, cmap='Greens')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        a.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=8,
               color='white' if cm[i, j] > cm.max() / 2 else 'black')
abrev = ['Flo', 'Herb', 'Silv', 'Agro', 'NVeg', 'Água']
a.set_xticks(range(N_CLASSES), abrev); a.set_yticks(range(N_CLASSES), abrev)
a.set_xlabel('U-Net'); a.set_ylabel('MapBiomas (referência)')
a.set_title(f'Matriz de confusão – blocos de validação (AG = {ag:.2f})')
plt.tight_layout(); plt.show()

In [ ]:
# Área de cada classe na bacia (1 pixel de 10 m = 100 m² = 0,01 ha)
contagem = np.bincount(mapa_unet[mapa_unet > 0], minlength=N_CLASSES + 1)[1:]
area = pd.DataFrame({'classe': NOMES, 'área (ha)': contagem * 0.01})
area['%'] = (area['área (ha)'] / area['área (ha)'].sum() * 100).round(1)
area.to_csv(f'{PASTA}/Area_classes_UNet_2025_Pirajibu.csv', index=False)
area.round(1)

## 9. Fechamento: qual método é melhor?

Ao longo do módulo usamos três caminhos para mapear a mesma bacia: o **produto pronto** (MapBiomas), a **classificação pixel a pixel** (Random Forest) e a **segmentação semântica** (U-Net).

**Qual é o melhor? Depende.** A escolha não é sobre o método mais moderno, e sim sobre qual atende melhor ao problema:

| Pergunta | Aponta para… |
|---|---|
| **Objetivo:** a legenda, a escala e o ano de um produto pronto respondem à minha pergunta? | Se sim, **MapBiomas** (ou outro produto validado). |
| **Objetivo:** as classes se separam pelo espectro, ou dependem de forma, textura e contexto? | Espectro → **Random Forest**. Forma e contexto → **U-Net**. |
| **Dados:** tenho apenas pontos de amostra, ou máscaras completas (rótulos densos)? | Pontos → **Random Forest**. Máscaras → **U-Net**. |
| **Dados:** qual a resolução da imagem? | Média (10–30 m) → RF costuma bastar. Alta → contexto espacial pesa mais. |
| **Habilidade e infraestrutura:** domino Python e tenho GPU, ou trabalho no GEE? | GEE, sem GPU → **Random Forest**. Python + GPU → **U-Net**. |
| **Uso:** é um mapa único ou um modelo que será reaplicado em muitas datas e áreas? | Mapa único → o método mais simples que resolve. Reaplicação → vale investir na **U-Net**. |

**Perguntas para a turma:**
1. Olhando os mapas desta prática e da aula de Random Forest, onde o contexto espacial parece ter feito diferença (bordas, estradas, pequenos corpos d'água)?
2. A U-Net aprendeu com o MapBiomas. Ela pode ser melhor que o MapBiomas? Por quê?
3. Para o problema ambiental do nosso projeto-base, qual dos três caminhos você escolheria, considerando seu objetivo, seus dados e suas habilidades?